In [2]:
import os
import sys
import json
import gzip
import pandas as pd

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)

!free -h
!df -h /content

In [3]:
!pip -q install "pandas==2.2.3" datasets huggingface_hub

In [4]:
import numpy as np
import pandas as pd
import pyarrow as pa
import datasets
import huggingface_hub

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("Datasets:", datasets.__version__)
print("Hugging Face:", huggingface_hub.__version__)

print("\nEnvironment ready!")

In [5]:
from datasets import load_dataset
from huggingface_hub import HfApi

REPO = "McAuley-Lab/Amazon-Reviews-2023"

info = HfApi().dataset_info(REPO, revision="2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e")

print("Dataset:", info.id)
print("Revision:", info.sha)
print("Connection successful!")

In [6]:
from huggingface_hub import HfApi, hf_hub_url
from datasets import load_dataset
from itertools import islice
import pandas as pd
import json
import re

api = HfApi()

REPO = "McAuley-Lab/Amazon-Reviews-2023"
revision = "2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e"

BASE_URL = (
    f"https://huggingface.co/datasets/"
    f"{REPO}/resolve/{revision}"
)

repo_files = api.list_repo_files(
    REPO,
    repo_type="dataset",
    revision=revision
)

meta_files = sorted([
    path for path in repo_files
    if path.startswith("raw_meta_Electronics/")
    and path.endswith(".parquet")
])

print("Revision:", revision)
print("Parquet files:", len(meta_files))

for path in meta_files:
    print(path)

if meta_files:
    meta_urls = [
        hf_hub_url(
            repo_id=REPO,
            filename=path,
            repo_type="dataset",
            revision=revision
        )
        for path in meta_files
    ]

    metadata = load_dataset(
        "parquet",
        data_files=meta_urls,
        split="train",
        streaming=True
    )

else:
    metadata = load_dataset(
        "json",
        data_files=(
            f"{BASE_URL}/raw/meta_categories/"
            "meta_Electronics.jsonl"
        ),
        split="train",
        streaming=True
    )

print("\nDataset connected!")
print(metadata)

In [7]:
sample = list(islice(metadata, 5))

df_sample = pd.DataFrame(sample)

print("Number of records:", len(sample))

print("\nAvailable columns:")
print(df_sample.columns.tolist())

print("\nData types:")
print(df_sample.dtypes)

display(df_sample)

print("\nFirst product detail:")

if sample:
    print(json.dumps(
        sample[0],
        indent=2,
        ensure_ascii=False,
        default=str
    ))

In [8]:
from collections import Counter
from itertools import islice
import pandas as pd
import json
import re

MAX_RECORDS = 100_000

PATTERNS = {
    "Headphones": r"\b(headphones?|earphones?|earbuds?|headsets?)\b",
    "Mouse": r"\b(mouse|mice)\b",
    "Keyboard": r"\bkeyboards?\b"
}

category_counts = Counter()
candidates = []
scanned = 0

for item in islice(metadata, MAX_RECORDS):

    scanned += 1

    title = str(item.get("title") or "")
    categories = item.get("categories") or []

    if isinstance(categories, str):
        categories = [categories]

    category_path = " > ".join(
        str(c) for c in categories
    )

    category_counts[category_path] += 1

    search_text = title + " " + category_path

    matched_groups = [
        group
        for group, pattern in PATTERNS.items()
        if re.search(
            pattern,
            search_text,
            flags=re.IGNORECASE
        )
    ]

    if matched_groups:

        record = dict(item)

        record["category_path"] = category_path
        record["leaf_category"] = (
            str(categories[-1])
            if categories else ""
        )
        record["candidate_groups"] = matched_groups

        candidates.append(record)

    if scanned % 10000 == 0:
        print(
            f"Scanned: {scanned:,} | "
            f"Candidates: {len(candidates):,}"
        )

candidate_df = pd.DataFrame(candidates)

print("\nTotal scanned:", scanned)
print("Total candidates:", len(candidate_df))

display(candidate_df.head(5))

In [9]:
taxonomy_rows = []

for item in candidates:

    for group in item["candidate_groups"]:

        taxonomy_rows.append({
            "candidate_group": group,
            "category_path": item["category_path"],
            "leaf_category": item["leaf_category"],
            "parent_asin": item.get("parent_asin"),
            "title": item.get("title")
        })

taxonomy_df = pd.DataFrame(
    taxonomy_rows,
    columns=[
        "candidate_group",
        "category_path",
        "leaf_category",
        "parent_asin",
        "title"
    ]
)

taxonomy_summary = (
    taxonomy_df
    .groupby([
        "candidate_group",
        "category_path",
        "leaf_category"
    ])
    .size()
    .reset_index(name="product_count")
    .sort_values(
        "product_count",
        ascending=False
    )
)

for group in PATTERNS:

    print(f"\n===== {group.upper()} =====")

    display(
        taxonomy_summary[
            taxonomy_summary["candidate_group"] == group
        ].head(20)
    )

In [10]:
for group in PATTERNS:

    subset = candidate_df[
        candidate_df["candidate_groups"].apply(
            lambda x: group in x
        )
    ]

    print(f"\n===== {group.upper()} =====")
    print("Total candidates:", len(subset))

    if subset.empty:
        continue

    display(
        subset[
            [
                "parent_asin",
                "title",
                "category_path",
                "price",
                "average_rating",
                "rating_number"
            ]
        ].sample(
            n=min(10, len(subset)),
            random_state=42
        )
    )

In [11]:
import math

def parse_price(value):
    try:
        number = float(value)

        if math.isfinite(number) and number > 0:
            return number

    except (ValueError, TypeError):
        pass

    return None


def parse_details(value):

    if isinstance(value, dict):
        return value

    if isinstance(value, str):

        try:
            result = json.loads(value)

            if isinstance(result, dict):
                return result

        except (ValueError, TypeError):
            pass

    return {}


def has_content(value):

    if value is None:
        return False

    if isinstance(value, (list, dict, tuple)):
        return len(value) > 0

    return str(value).strip().lower() not in {
        "", "none", "nan", "null", "[]", "{}"
    }


def has_image(value):

    if not isinstance(value, dict):
        return False

    for key in ["hi_res", "large", "thumb"]:

        urls = value.get(key) or []

        if isinstance(urls, str):
            urls = [urls]

        if any(
            isinstance(url, str)
            and url.startswith(("https://", "http://"))
            for url in urls
        ):
            return True

    return False


candidate_df["price_usd"] = (
    candidate_df["price"].apply(parse_price)
)

candidate_df["details_parsed"] = (
    candidate_df["details"].apply(parse_details)
)

candidate_df["has_price"] = (
    candidate_df["price_usd"].notna()
)

candidate_df["has_features"] = (
    candidate_df["features"].apply(has_content)
)

candidate_df["has_description"] = (
    candidate_df["description"].apply(has_content)
)

candidate_df["has_details"] = (
    candidate_df["details_parsed"].apply(bool)
)

candidate_df["has_brand"] = (
    candidate_df["details_parsed"].apply(
        lambda d: has_content(d.get("Brand"))
    )
)

candidate_df["has_image"] = (
    candidate_df["images"].apply(has_image)
)

expanded_df = candidate_df.explode(
    "candidate_groups"
)

metrics = [
    "has_price",
    "has_features",
    "has_description",
    "has_details",
    "has_brand",
    "has_image"
]

coverage_df = (
    expanded_df
    .groupby("candidate_groups")[metrics]
    .mean()
    .mul(100)
    .round(2)
)

coverage_df.insert(
    0,
    "candidate_count",
    expanded_df.groupby("candidate_groups").size()
)

display(coverage_df)

Phân loại lại sản phẩm

In [12]:
CATEGORY_MAP = {
    "Headphones": {
        "earbud headphones",
        "over-ear headphones",
        "on-ear headphones",
        "headphones & earbuds",
        "computer headsets"
    },
    "Mouse": {
        "mice"
    },
    "Keyboard": {
        "keyboards"
    }
}

EXCLUSION_PATTERNS = [
    r"\breplacement\s+(earpads?|ear pads?|cables?)\b",
    r"\bmouse\s?pads?\b",
    r"\bmouse\s+(mover|jiggler)\b",
    r"\bkeyboard\s+(cover|skin|protector)\b",
    r"\bkeycap\s+sets?\b",
    r"\bkeycaps?\s+replacement\b",
    r"\bkeyboard\s+and\s+mouse\s+(combo|set|bundle)\b",
    r"\bkeyboard\s*(?:&|\+|/)\s*mouse\b",
    r"\bkeyboard\s+mouse\s+combo\b",
    r"\bheadphone\s+(stand|holder|case)\b",
    r"\bearbud\s+(case|cover)\b",
    r"\bcharging\s+case\s+for\b",
    r"\bfpv\s+(headset|goggles)\b"
]

def classify_strict(item):

    leaf = str(
        item.get("leaf_category") or ""
    ).strip().lower()

    title = str(
        item.get("title") or ""
    ).strip().lower()

    path = str(
        item.get("category_path") or ""
    ).strip().lower()

    pid = item.get("parent_asin")

    if not title or not pid:
        return None

    if not path.startswith("electronics"):
        return None

    if any(
        re.search(pattern, title)
        for pattern in EXCLUSION_PATTERNS
    ):
        return None

    for category, allowed_leaves in CATEGORY_MAP.items():

        if leaf in allowed_leaves:
            return category

    return None


candidate_df["smartbuy_category"] = (
    candidate_df.apply(
        classify_strict,
        axis=1
    )
)

clean_df = candidate_df[
    candidate_df["smartbuy_category"].notna()
].copy()

clean_df = clean_df.drop_duplicates(
    subset=["parent_asin"]
).reset_index(drop=True)

print("Original candidates:", len(candidate_df))
print("Strict candidates:", len(clean_df))
print("Excluded or deferred:",
      len(candidate_df) - len(clean_df))

display(
    clean_df["smartbuy_category"].value_counts()
)

Tính lại Data Coverage sau phân loại

In [13]:
clean_metrics = [
    "has_price",
    "has_features",
    "has_description",
    "has_details",
    "has_brand",
    "has_image"
]

clean_coverage = (
    clean_df
    .groupby("smartbuy_category")[clean_metrics]
    .mean()
    .mul(100)
    .round(2)
)

clean_coverage.insert(
    0,
    "product_count",
    clean_df.groupby(
        "smartbuy_category"
    ).size()
)

clean_coverage.insert(
    1,
    "products_with_price",
    clean_df.groupby(
        "smartbuy_category"
    )["has_price"].sum()
)

display(clean_coverage)

In [14]:
import matplotlib.pyplot as plt

before = coverage_df[clean_metrics].copy()
after = clean_coverage[clean_metrics].copy()

for category in [
    "Headphones",
    "Mouse",
    "Keyboard"
]:

    if category not in after.index:
        continue

    comparison = pd.DataFrame({
        "Before cleaning":
            before.loc[category],
        "After classification":
            after.loc[category]
    })

    print(f"\n===== {category} =====")

    display(comparison)

    comparison.plot(
        kind="bar",
        figsize=(10, 4)
    )

    plt.title(f"{category} - Data Coverage")
    plt.ylabel("Coverage (%)")
    plt.ylim(0, 100)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## **Kiểm tra khả năng xây dựng catalog MVP**

In [15]:
priced_df = clean_df[
    clean_df["has_price"]
].copy()

mvp_candidates = priced_df[
    priced_df["has_image"]
    & priced_df["has_details"]
].copy()

summary = []

for category in [
    "Headphones",
    "Mouse",
    "Keyboard"
]:

    total = clean_df[
        clean_df["smartbuy_category"] == category
    ]

    priced = priced_df[
        priced_df["smartbuy_category"] == category
    ]

    eligible = mvp_candidates[
        mvp_candidates["smartbuy_category"] == category
    ]

    summary.append({
        "category": category,
        "strict_candidates": len(total),
        "valid_price": len(priced),
        "basic_mvp_candidates": len(eligible),
        "minimum_target": 30,
        "metadata_quantity_gate": (
            "PASS" if len(eligible) >= 30
            else "FAIL"
        )
    })

mvp_summary = pd.DataFrame(summary)

display(mvp_summary)

In [16]:
#Kiểm tra thủ công
audit_columns = [
    "parent_asin",
    "smartbuy_category",
    "title",
    "category_path",
    "price_usd",
    "average_rating",
    "rating_number",
    "features",
    "details"
]

audit_samples = []

for category in [
    "Headphones",
    "Mouse",
    "Keyboard"
]:

    subset = clean_df[
        clean_df["smartbuy_category"] == category
    ]

    if subset.empty:
        continue

    group_sample = subset.sample(
        n=min(30, len(subset)),
        random_state=42
    )

    audit_samples.append(group_sample)

manual_audit = pd.concat(
    audit_samples,
    ignore_index=True
)[audit_columns].copy()

manual_audit["category_correct"] = ""
manual_audit["is_accessory"] = ""
manual_audit["is_combo"] = ""
manual_audit["notes"] = ""

pd.set_option("display.max_colwidth", 150)

display(
    manual_audit[
        [
            "smartbuy_category",
            "title",
            "price_usd",
            "category_path"
        ]
    ]
)

manual_audit.to_csv(
    "/content/smartbuy_manual_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

In [17]:
from collections import Counter

attribute_key_rows = []

for category in [
    "Headphones",
    "Mouse",
    "Keyboard"
]:

    subset = mvp_candidates[
        mvp_candidates["smartbuy_category"] == category
    ]

    key_counts = Counter()

    for details in subset["details_parsed"]:

        if not isinstance(details, dict):
            continue

        valid_keys = {
            str(key).strip().lower()
            for key, value in details.items()
            if has_content(value)
        }

        key_counts.update(valid_keys)

    for key, count in key_counts.items():

        attribute_key_rows.append({
            "category": category,
            "attribute": key,
            "product_count": count,
            "coverage_pct": round(
                count / len(subset) * 100, 2
            )
        })

attribute_keys_df = pd.DataFrame(
    attribute_key_rows
)

for category in [
    "Headphones",
    "Mouse",
    "Keyboard"
]:

    print(f"\n===== {category} =====")

    display(
        attribute_keys_df[
            attribute_keys_df["category"] == category
        ].sort_values(
            "coverage_pct",
            ascending=False
        ).head(40)
    )

In [18]:
audit_fields = [
    "parent_asin",
    "smartbuy_category",
    "title",
    "category_path",
    "price_usd",
    "features",
    "details"
]

priced_audit = pd.concat([
    group.sample(
        n=min(30, len(group)),
        random_state=42
    )

    for _, group in mvp_candidates.groupby(
        "smartbuy_category"
    )

], ignore_index=True)[audit_fields].copy()

priced_audit["is_correct_product"] = ""
priced_audit["is_accessory"] = ""
priced_audit["is_combo"] = ""
priced_audit["notes"] = ""

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

display(
    priced_audit[
        [
            "parent_asin",
            "smartbuy_category",
            "title",
            "price_usd"
        ]
    ]
)

priced_audit.to_csv(
    "/content/smartbuy_priced_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

In [19]:
review_files = sorted([
    path for path in repo_files

    if path.startswith("raw_review_Electronics/")
    and path.endswith(".parquet")
])

print("Review Parquet files:", len(review_files))

if review_files:

    review_urls = [
        hf_hub_url(
            repo_id=REPO,
            filename=path,
            repo_type="dataset",
            revision=revision
        )
        for path in review_files
    ]

    reviews = load_dataset(
        "parquet",
        data_files=review_urls,
        split="train",
        streaming=True
    )

else:

    reviews = load_dataset(
        "json",
        data_files=(
            f"{BASE_URL}/raw/review_categories/"
            "Electronics.jsonl"
        ),
        split="train",
        streaming=True
    )

review_sample = list(islice(reviews, 5))

review_sample_df = pd.DataFrame(
    review_sample
)

print("Review fields:")
print(review_sample_df.columns.tolist())

display(
    review_sample_df.drop(
        columns=["user_id"],
        errors="ignore"
    )
)

In [20]:
from collections import defaultdict
from itertools import islice
import hashlib
import json
import pandas as pd

MAX_REVIEWS = 2_000_000
MAX_PER_PRODUCT = 20

product_ids = set(
    mvp_candidates["parent_asin"]
)

review_pool = defaultdict(list)
seen_reviews = defaultdict(set)

scanned_reviews = 0

for review in islice(reviews, MAX_REVIEWS):

    scanned_reviews += 1

    pid = review.get("parent_asin")
    text = review.get("text")

    if (
        pid in product_ids
        and isinstance(text, str)
        and text.strip()
        and review.get("rating") in [1, 2, 3, 4, 5]
        and len(review_pool[pid]) < MAX_PER_PRODUCT
    ):

        fingerprint = hashlib.sha256(
            json.dumps(
                [
                    pid,
                    review.get("asin"),
                    review.get("user_id"),
                    review.get("timestamp"),
                    text
                ],
                ensure_ascii=False,
                default=str
            ).encode("utf-8")
        ).hexdigest()

        if fingerprint not in seen_reviews[pid]:

            seen_reviews[pid].add(fingerprint)

            review_pool[pid].append({
                "review_id": fingerprint,
                "parent_asin": pid,
                "asin": review.get("asin"),
                "rating": review.get("rating"),
                "title": review.get("title"),
                "text": text,
                "timestamp": review.get("timestamp"),
                "verified_purchase":
                    review.get("verified_purchase"),
                "helpful_vote":
                    review.get("helpful_vote")
            })

    if scanned_reviews % 100_000 == 0:

        print(
            f"Scanned: {scanned_reviews:,} | "
            f"Matched products: {len(review_pool):,} | "
            f"Saved reviews: "
            f"{sum(map(len, review_pool.values())):,}"
        )

print("\nREVIEW MATCHING COMPLETED")
print("Scanned reviews:", scanned_reviews)
print("Matched products:", len(review_pool))

Scanned: 100,000 | Matched products: 368 | Saved reviews: 1,094
Scanned: 200,000 | Matched products: 505 | Saved reviews: 1,878
Scanned: 300,000 | Matched products: 577 | Saved reviews: 2,515
Scanned: 400,000 | Matched products: 632 | Saved reviews: 2,983
Scanned: 500,000 | Matched products: 678 | Saved reviews: 3,464
Scanned: 600,000 | Matched products: 713 | Saved reviews: 3,792
Scanned: 700,000 | Matched products: 737 | Saved reviews: 4,104
Scanned: 800,000 | Matched products: 777 | Saved reviews: 4,420
Scanned: 900,000 | Matched products: 800 | Saved reviews: 4,689
Scanned: 1,000,000 | Matched products: 820 | Saved reviews: 4,995
Scanned: 1,100,000 | Matched products: 849 | Saved reviews: 5,289
Scanned: 1,200,000 | Matched products: 869 | Saved reviews: 5,569
Scanned: 1,300,000 | Matched products: 890 | Saved reviews: 5,830
Scanned: 1,400,000 | Matched products: 900 | Saved reviews: 6,009
Scanned: 1,500,000 | Matched products: 921 | Saved reviews: 6,224
Scanned: 1,600,000 | Matched

In [21]:
mvp_review_df = mvp_candidates.copy()

mvp_review_df["retained_review_count"] = (
    mvp_review_df["parent_asin"]
    .map(lambda pid: len(review_pool.get(pid, [])))
)

review_coverage_rows = []

for category, group in mvp_review_df.groupby(
    "smartbuy_category"
):

    count = len(group)

    with_review = int(
        group["retained_review_count"].ge(1).sum()
    )

    with_10_reviews = int(
        group["retained_review_count"].ge(10).sum()
    )

    with_20_reviews = int(
        group["retained_review_count"].ge(20).sum()
    )

    review_coverage_rows.append({
        "category": category,
        "total_products": count,
        "products_with_review": with_review,
        "products_with_10_reviews": with_10_reviews,
        "products_with_20_reviews": with_20_reviews,
        "coverage_review_pct": round(
            with_review / count * 100, 2
        ),
        "coverage_10_reviews_pct": round(
            with_10_reviews / count * 100, 2
        ),
        "retained_reviews":
            group["retained_review_count"].sum()
    })

review_coverage_df = pd.DataFrame(
    review_coverage_rows
)

display(review_coverage_df)

In [22]:
review_eligible_df = mvp_review_df[
    mvp_review_df["retained_review_count"] >= 10
].copy()

TARGET_AUDIT = {
    "Headphones": 120,
    "Mouse": 70,
    "Keyboard": 150
}

audit_groups = []

for category, target in TARGET_AUDIT.items():

    subset = review_eligible_df[
        review_eligible_df["smartbuy_category"]
        == category
    ]

    if not subset.empty:

        audit_groups.append(
            subset.sample(
                n=min(target, len(subset)),
                random_state=42
            )
        )

audit_fields = [
    "parent_asin",
    "smartbuy_category",
    "title",
    "category_path",
    "price_usd",
    "features",
    "description",
    "details",
    "retained_review_count"
]

if audit_groups:

    final_audit = pd.concat(
        audit_groups,
        ignore_index=True
    )[audit_fields].copy()

else:

    final_audit = pd.DataFrame(
        columns=audit_fields
    )

final_audit["audit_status"] = "PENDING"
final_audit["audit_reason"] = ""

display(
    final_audit.groupby(
        "smartbuy_category"
    ).size()
)

display(
    final_audit[
        [
            "parent_asin",
            "smartbuy_category",
            "title",
            "price_usd",
            "retained_review_count",
            "audit_status"
        ]
    ]
)

final_audit.to_csv(
    "/content/smartbuy_final_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

## Pain Mining — bằng chứng trực tiếp trước khi mua

Mẫu cố định 200 review từ `review_pool`, seed 42, không chọn theo rating hay từ khóa. Mỗi review là một đơn vị đếm, không phải một người. Kiểm tra ngành hàng thực tế qua title/category; loại phụ kiện, combo. Gắn `YES` nếu review trực tiếp mô tả khó nghiên cứu/so sánh, thiếu hoặc hiểu sai thông tin trước mua, hoặc mua nhầm vì thông tin trước mua; `NO` nếu chỉ phàn nàn chất lượng sau mua, lỗi vận chuyển, hoặc không có bằng chứng trực tiếp; `OUT_SCOPE` nếu sai ngành. Với YES, điền pain_type, quote (đoạn nguyên văn ngắn ≤20 từ, không chứa dữ liệu cá nhân), reason. Không sửa sample hoặc nội dung review. PENDING không được tính là NO. Chỉ công bố số tổng hợp và đoạn trích ngắn; không commit CSV thô.

Đây là mẫu khám phá từ prefix Electronics, tối đa 20 review/sản phẩm; không đại diện cho tất cả khách hàng. Ngưỡng BTC cần kiểm tra riêng.

In [23]:
import pandas as pd, hashlib, json, re
from pathlib import Path
from google.colab import files
assert 'review_pool' in globals() and 'mvp_candidates' in globals(), 'Chạy các cell Data Discovery trước.'
pain_products = mvp_candidates[['parent_asin','smartbuy_category','title','category_path']].drop_duplicates('parent_asin').rename(columns={'title':'product_title'})
pain_all = pd.DataFrame([r for rs in review_pool.values() for r in rs]).drop_duplicates('review_id')
pain_all = pain_all.merge(pain_products, on='parent_asin', how='inner', validate='many_to_one')
pain_all = pain_all[pain_all['text'].fillna('').str.strip().ne('')].sort_values('review_id').reset_index(drop=True)
assert len(pain_all)>0
PAIN_SEED=42
PAIN_SAMPLE_SIZE=200
pain_sample = pain_all.sample(n=min(PAIN_SAMPLE_SIZE,len(pain_all)),random_state=PAIN_SEED).sort_values('review_id').reset_index(drop=True)
pain_snapshot = hashlib.sha256('\n'.join(pain_sample.review_id).encode()).hexdigest()
pain_cols=['review_id','parent_asin','asin','smartbuy_category','product_title','category_path','rating','title','text','timestamp']
pain_annotation = pain_sample[pain_cols].copy()
pain_annotation['label']='PENDING'
pain_annotation['pain_type']=''
pain_annotation['quote']=''
pain_annotation['reason']=''
pain_annotation['reviewer']=''
print('Tổng review hợp lệ trong tập ứng viên:',len(pain_all))
print('Mẫu cố định:',len(pain_sample),'seed:',PAIN_SEED,'hash:',pain_snapshot)
display(pain_sample.smartbuy_category.value_counts())
pain_annotation.to_csv('/content/pain_annotation.csv',index=False,encoding='utf-8-sig')
files.download('/content/pain_annotation.csv')


Tổng review hợp lệ trong tập ứng viên: 7173
Mẫu cố định: 200 seed: 42 hash: e512e1d484cb6d94d6fbbf796d4e06cf3d014f00fc5cb8a6f7eba19e78ea09cd


smartbuy_category
Headphones    144
Keyboard       29
Mouse          27
Name: count, dtype: int64

### Gán nhãn và kiểm tra
Mở CSV, điền label YES / NO / OUT_SCOPE cho tất cả dòng. YES điền loại pain: P01 (thiếu/khó hiểu thông tin), P02 (kỳ vọng sai do thông tin trước mua), P03 (mua nhầm vì thông tin tương thích), P04 (khó so sánh/chọn), P05 (mô tả gây hiểu nhầm); quote nguyên văn ≤20 từ, reason giải thích liên hệ với trước mua; reviewer là mã thành viên. NO/OUT_SCOPE cũng ghi reason. Không đổi review_id, không thêm/xóa dòng. Nên review chéo YES và trường hợp không rõ. Upload CSV đã gán nhãn ở cell dưới.


In [24]:
uploaded = files.upload()
assert len(uploaded)==1, 'Chỉ upload một CSV gán nhãn.'
import io
labels = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))),dtype=str).fillna('')
assert set(['review_id','label','pain_type','quote','reason','reviewer']).issubset(labels.columns)
assert labels.review_id.is_unique and set(labels.review_id)==set(pain_sample.review_id), 'Mẫu không khớp, không được thêm/xóa dòng.'
labels['label']=labels.label.str.strip().str.upper()
assert labels.label.isin(['YES','NO','OUT_SCOPE']).all(), 'Còn dòng chưa gán nhãn.'
assert labels.reason.str.strip().ne('').all() and labels.reviewer.str.strip().ne('').all(), 'Cần lý do và mã reviewer cho mọi dòng.'
result = pain_sample.merge(labels[['review_id','label','pain_type','quote','reason','reviewer']], on='review_id',validate='one_to_one')
positives=result[result.label=='YES'].copy()
for _,r in positives.iterrows():
    q=r.quote.strip()
    assert r.pain_type in ['P01','P02','P03','P04','P05'], 'Pain type không hợp lệ.'
    assert q and len(q.split())<=20 and (q in str(r.text) or q in str(r.title)), f'Quote phải nguyên văn, ngắn: {r.review_id}'
N=int(result.label.ne('OUT_SCOPE').sum()); x=len(positives)
assert N>0, 'Không có review đúng phạm vi.'
pct=100*x/N
print(f'Pain: {x}/{N} = {pct:.2f}% (loại {len(result)-N} review ngoài scope)')
print('Số ví dụ đủ nguồn:',len(positives), '| Đủ ≥5:',len(positives)>=5)
display(result.groupby(['smartbuy_category','label']).size().unstack(fill_value=0))
print('Cần đối chiếu ngưỡng BTC trước khi đánh dấu đạt.')
def safe(s): return str(s).replace('|','\\|').replace('\n',' ').replace('\r',' ')
lines=['# Data mining — SmartBuy','', 'Nguồn: Amazon Reviews 2023, Electronics; revision '+str(revision)+'.', 'https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023', '',f'Mẫu cố định {len(result)} review, seed {PAIN_SEED}, SHA256 danh sách ID: {pain_snapshot}.',f'Tập đầu vào: {len(pain_all)} review giữ lại; quét prefix, tối đa 20 review mỗi sản phẩm.',f'Loại ngoài scope: {len(result)-N}. N={N} review đúng scope; x={x} review có pain trực tiếp; tỷ lệ {pct:.2f}%.', '', 'Quy tắc: YES chỉ khi có căn cứ trực tiếp về khó khăn nghiên cứu/chọn trước mua. Không đếm lỗi sau mua đơn thuần. Mỗi review đếm tối đa một lần. Gán nhãn thủ công; tái lập mẫu và phép đếm từ notebook và bảng nhãn nội bộ.', '', '| ID review | ASIN | Pain | Trích dẫn ngắn | Lý do |','|---|---|---|---|---|']
for _,r in positives.head(5).iterrows():
    lines.append('| '+' | '.join(safe(v) for v in [r.review_id,r.asin,r.pain_type,r.quote,r.reason])+' |')
lines+=['','Giới hạn: mẫu từ tập review đã thu thập, không đại diện mọi khách hàng; tỷ lệ là tỷ lệ review, không phải tỷ lệ người mua.','Trạng thái: '+('Có ≥5 ví dụ; chờ đối chiếu ngưỡng BTC.' if x>=5 else 'Chưa đủ 5 ví dụ; không đánh dấu đạt.')]
Path('/content/data-mining.md').write_text('\n'.join(lines),encoding='utf-8')
files.download('/content/data-mining.md')


Saving pain_annotation_labeled.csv to pain_annotation_labeled.csv
Pain: 13/181 = 7.18% (loại 19 review ngoài scope)
Số ví dụ đủ nguồn: 13 | Đủ ≥5: True


label               NO  OUT_SCOPE  YES
smartbuy_category                     
Headphones         123         13    8
Keyboard            21          4    4
Mouse               24          2    1

Cần đối chiếu ngưỡng BTC trước khi đánh dấu đạt.
